# 09 Weekly Challenge

전체 프로세스: Qwen Model → LoRA/QLoRA → PTQ → GGUF, Llama.cpp

## 진행 순서
- Phase 0. Baseline 확보
- Phase 1. Fine-Tuning: LoRA vs QLoRA
- Phase 2. Post-Training Quantization (PTQ)
- Phase 3. GGUF 변환 및 Llama.cpp 추론
- 최종 정리 (표 1, 표 2)

## Phase 0. Baseline 확보

- 0-1. 환경 설정 및 라이브러리 import
- 0-2. Qwen 원본 모델 로드 (bfloat16)
- 0-3. 샘플 prompt 추론 테스트
- 0-4. Baseline metrics 기록 (perplexity, memory, latency)
- 0-5. [Empty Cache] 원본 모델 메모리 해제

In [1]:
# TensorFlow 비활성화
import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# 0-1. 환경 설정 및 라이브러리 import
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device: {device}")

# 전체 Phase에 걸쳐 metrics를 누적할 딕셔너리 (표 1 원본 데이터)
performance_metrics_by_phase = {}

device: cuda


In [2]:
# 0-2. Qwen 원본 모델 로드 (bfloat16)
model_name_or_path = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

original_qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    dtype=torch.bfloat16
).to(device)

original_qwen_model.eval()
print(f"model loaded: {model_name_or_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [3]:
# 0-3. 샘플 prompt 추론 테스트
sample_prompt = "다음 주 화요일 오후 3시에 회의 일정을 잡아줘."

input_token_ids = tokenizer(sample_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    generated_token_ids = original_qwen_model.generate(
        **input_token_ids,
        max_new_tokens=100,
        do_sample=False
    )

generated_text = tokenizer.decode(generated_token_ids[0], skip_special_tokens=True)
print(generated_text)

다음 주 화요일 오후 3시에 회의 일정을 잡아줘. 그리고 그날의 날씨를 알려줘.
주말에는 휴식을 취하고, 다음 주 화요일은 회의가 필요하므로, 아래와 같이 회의 일정을 잡겠습니다.

1. 화요일 오후 3시: 회의

2. 화요일 저녁: 휴식 시간

이번 주 화요일은 회의가 예정되어 있으니, 이 기간 동안


In [4]:
# 0-4. Baseline metrics 기록 (perplexity, memory, latency)
def measure_memory_usage_in_megabytes(model):
    total_parameter_bytes = sum(
        parameter.element_size() * parameter.numel()
        for parameter in model.parameters()
    )
    return total_parameter_bytes / (1024 ** 2)


def measure_inference_latency_in_seconds(model, tokenizer, prompt_text, device, number_of_runs=5):
    input_token_ids = tokenizer(prompt_text, return_tensors="pt").to(device)

    # warm-up (초기 실행 오버헤드 제외)
    with torch.no_grad():
        model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)

    elapsed_time_list = []
    for _ in range(number_of_runs):
        start_time = time.time()
        with torch.no_grad():
            model.generate(**input_token_ids, max_new_tokens=50, do_sample=False)
        elapsed_time_list.append(time.time() - start_time)

    return sum(elapsed_time_list) / len(elapsed_time_list)


def measure_perplexity(model, tokenizer, evaluation_text_list, device):
    total_negative_log_likelihood = 0.0
    total_token_count = 0

    for evaluation_text in evaluation_text_list:
        input_token_ids = tokenizer(evaluation_text, return_tensors="pt").to(device)
        with torch.no_grad():
            model_output = model(**input_token_ids, labels=input_token_ids["input_ids"])

        token_count = input_token_ids["input_ids"].size(1)
        total_negative_log_likelihood += model_output.loss.item() * token_count
        total_token_count += token_count

    average_negative_log_likelihood = total_negative_log_likelihood / total_token_count
    return torch.exp(torch.tensor(average_negative_log_likelihood)).item()

In [5]:
evaluation_text_list = [
    "다음 주 화요일 오후 3시에 회의 일정을 잡아줘.",
    "이번 주 금요일에 잡힌 일정이 있는지 확인해줘.",
    "내일 오전 10시에 팀 미팅 일정을 추가해줘.",
    "다음 달 첫째 주에 워크숍 일정을 등록해줘.",
    "오늘 오후 6시 저녁 약속을 캘린더에 저장해줘.",
    "이번 주 수요일 일정을 모두 삭제해줘.",
    "다음 주 월요일부터 금요일까지 매일 아침 회의를 잡아줘.",
    "이번 달 마지막 주에 휴가 일정을 등록해줘.",
    "내일 오후 2시로 잡힌 미팅을 오후 4시로 변경해줘.",
    "이번 주말에 잡힌 일정이 있으면 알려줘.",
    "다음 주 목요일 점심 약속을 취소해줘.",
    "매주 화요일 오전 9시에 반복 일정을 등록해줘.",
    "이번 주 일정 중 가장 빠른 일정이 무엇인지 알려줘.",
    "다음 주 출장 일정을 캘린더에 추가해줘.",
    "오늘 저녁에 잡힌 약속 시간을 확인해줘.",
    "이번 달 중 비어있는 날짜를 찾아줘.",
    "다음 주 화요일 회의를 다른 요일로 옮겨줘.",
    "이번 주 금요일 오후 일정을 모두 보여줘.",
    "내일 오전 회의 참석자 명단을 확인해줘.",
    "다음 주에 예정된 모든 일정을 요약해줘.",
]

performance_metrics_by_phase["baseline"] = {
    "memory_mb": measure_memory_usage_in_megabytes(original_qwen_model),
    "latency_sec": measure_inference_latency_in_seconds(
        original_qwen_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        original_qwen_model, tokenizer, evaluation_text_list, device
    ),
}

print(performance_metrics_by_phase["baseline"])

{'memory_mb': 2944.4013671875, 'latency_sec': 1.6323445320129395, 'perplexity': 13.685714721679688}


In [6]:
# 0-5. [Empty Cache] 원본 모델 메모리 해제
def empty_device_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    # 둘 다 없으면 (cpu) 아무것도 하지 않음

gc.collect()
empty_device_cache()

### Phase 0 실행 현황

- 실행 환경: VSCode ↔ Colab 원격 연결, GPU 런타임 (CUDA, T4)
- device 분기 및 empty cache 함수를 CUDA/MPS/CPU 대응 버전으로 수정 후 정상 동작 확인
- Baseline 확정 (Qwen2.5-1.5B-Instruct, bfloat16 기준)
  - memory_mb: 2944.40
  - latency_sec: 2.10 (50 tokens 생성 기준)
  - perplexity: 13.66 (evaluation 문장 20개 기준)
- Phase 1(LoRA/QLoRA) 진행을 위한 baseline 값으로 확정

---

## Phase 1. Fine-Tuning: LoRA vs QLoRA

- 1-1. Dataset 준비 (DaySync 도메인 데이터)
- 1-2. LoRA
- 1-3. QLoRA
- 1-4. LoRA vs QLoRA 비교 및 선택

In [7]:
# 1-1. Dataset 
# 1-1-1. Train / Validation Split (19 / 5)
instruction_response_pair_list = [
    {
        "instruction": "다음 주 화요일 오후 3시에 회의 일정을 잡아줘.",
        "response": "다음 주 화요일 오후 3시에 회의 일정을 등록했습니다."
    },
    {
        "instruction": "이번 주 금요일에 잡힌 일정이 있는지 확인해줘.",
        "response": "이번 주 금요일에 등록된 일정은 오후 2시 팀 회의입니다."
    },
    {
        "instruction": "내일 오전 10시에 팀 미팅 일정을 추가해줘.",
        "response": "내일 오전 10시에 팀 미팅 일정을 추가했습니다."
    },
    {
        "instruction": "다음 달 첫째 주에 워크숍 일정을 등록해줘.",
        "response": "다음 달 첫째 주 월요일에 워크숍 일정을 등록했습니다."
    },
    {
        "instruction": "오늘 오후 6시 저녁 약속을 캘린더에 저장해줘.",
        "response": "오늘 오후 6시 저녁 약속을 캘린더에 저장했습니다."
    },
    {
        "instruction": "이번 주 수요일 일정을 모두 삭제해줘.",
        "response": "이번 주 수요일에 등록된 일정을 모두 삭제했습니다."
    },
    {
        "instruction": "다음 주 월요일부터 금요일까지 매일 아침 회의를 잡아줘.",
        "response": "다음 주 월요일부터 금요일까지 매일 아침 9시에 회의 일정을 등록했습니다."
    },
    {
        "instruction": "이번 달 마지막 주에 휴가 일정을 등록해줘.",
        "response": "이번 달 마지막 주 월요일부터 금요일까지 휴가 일정을 등록했습니다."
    },
    {
        "instruction": "내일 오후 2시로 잡힌 미팅을 오후 4시로 변경해줘.",
        "response": "내일 미팅 일정을 오후 2시에서 오후 4시로 변경했습니다."
    },
    {
        "instruction": "이번 주말에 잡힌 일정이 있으면 알려줘.",
        "response": "이번 주말에는 등록된 일정이 없습니다."
    },
    {
        "instruction": "다음 주 목요일 점심 약속을 취소해줘.",
        "response": "다음 주 목요일 점심 약속을 취소했습니다."
    },
    {
        "instruction": "매주 화요일 오전 9시에 반복 일정을 등록해줘.",
        "response": "매주 화요일 오전 9시 반복 일정을 등록했습니다."
    },
    {
        "instruction": "이번 주 일정 중 가장 빠른 일정이 무엇인지 알려줘.",
        "response": "이번 주 가장 빠른 일정은 월요일 오전 9시 회의입니다."
    },
    {
        "instruction": "다음 주 출장 일정을 캘린더에 추가해줘.",
        "response": "다음 주 수요일부터 목요일까지 출장 일정을 캘린더에 추가했습니다."
    },
    {
        "instruction": "오늘 저녁에 잡힌 약속 시간을 확인해줘.",
        "response": "오늘 저녁 약속은 오후 7시입니다."
    },
    {
        "instruction": "이번 달 중 비어있는 날짜를 찾아줘.",
        "response": "이번 달 중 15일과 22일이 비어있습니다."
    },
    {
        "instruction": "다음 주 화요일 회의를 다른 요일로 옮겨줘.",
        "response": "다음 주 화요일 회의를 수요일로 옮겼습니다."
    },
    {
        "instruction": "이번 주 금요일 오후 일정을 모두 보여줘.",
        "response": "이번 주 금요일 오후 일정은 3시 회의, 5시 보고입니다."
    },
    {
        "instruction": "내일 오전 회의 참석자 명단을 확인해줘.",
        "response": "내일 오전 회의 참석자는 총 5명입니다."
    },
    {
        "instruction": "다음 주에 예정된 모든 일정을 요약해줘.",
        "response": "다음 주에는 회의 3건, 출장 1건, 워크숍 1건이 예정되어 있습니다."
    },
    {
        "instruction": "이번 주 화요일 회의 시간을 알려줘.",
        "response": "이번 주 화요일 회의는 오후 3시입니다."
    },
    {
        "instruction": "다음 주 수요일 오전 일정을 비워줘.",
        "response": "다음 주 수요일 오전 일정을 모두 비웠습니다."
    },
    {
        "instruction": "오늘 일정 중 취소된 항목이 있는지 알려줘.",
        "response": "오늘 일정 중 취소된 항목은 없습니다."
    },
    {
        "instruction": "이번 주 목요일에 새로운 회의를 추가해줘.",
        "response": "이번 주 목요일 오후 1시에 새로운 회의를 추가했습니다."
    }
]

print(f"total pairs: {len(instruction_response_pair_list)}")

# 1-1-2. Prompt Template 구성 (Qwen Instruct format)
train_pair_list = instruction_response_pair_list[:19]
validation_pair_list = instruction_response_pair_list[19:]

print(f"train: {len(train_pair_list)}, validation: {len(validation_pair_list)}")

# 1-1-3. Prompt Template 구성 (Qwen Instruct format)
"""
Qwen2.5-Instruct는 ChatML 형식(chat template)을 사용하므로, tokenizer.apply_chat_template으로 변환.
"""
def build_chat_formatted_text(instruction_text, response_text, tokenizer):
    message_list = [
        {"role": "user", "content": instruction_text},
        {"role": "assistant", "content": response_text}
    ]
    return tokenizer.apply_chat_template(message_list, tokenize=False)

formatted_train_text_list = [
    build_chat_formatted_text(pair["instruction"], pair["response"], tokenizer)
    for pair in train_pair_list
]

print(formatted_train_text_list[0])

# 1-1-4. Dataset 객체 변환 (Hugging Face datasets)
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": formatted_train_text_list})

formatted_validation_text_list = [
    build_chat_formatted_text(pair["instruction"], pair["response"], tokenizer)
    for pair in validation_pair_list
]
validation_dataset = Dataset.from_dict({"text": formatted_validation_text_list})

print(train_dataset)
print(validation_dataset)

total pairs: 24
train: 19, validation: 5
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
다음 주 화요일 오후 3시에 회의 일정을 잡아줘.<|im_end|>
<|im_start|>assistant
다음 주 화요일 오후 3시에 회의 일정을 등록했습니다.<|im_end|>

Dataset({
    features: ['text'],
    num_rows: 19
})
Dataset({
    features: ['text'],
    num_rows: 5
})


In [8]:
%pip install -U torchao --break-system-packages

# 1-2. LoRA
## 1-2-1. 기반 모델 로드 (bfloat16)
lora_base_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    dtype=torch.bfloat16
).to(device)

print(f"lora base model loaded on: {next(lora_base_model.parameters()).device}")

## 1-2-2. LoRA Config 설정 (rank, alpha, target_modules)
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

lora_model = get_peft_model(lora_base_model, lora_config)
lora_model.print_trainable_parameters()

## 1-2-3. 학습 실행
def tokenize_function(example_batch):
    tokenized_output = tokenizer(
        example_batch["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    tokenized_output["labels"] = tokenized_output["input_ids"].copy()
    return tokenized_output


tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True)

from transformers import TrainingArguments, Trainer

lora_training_arguments = TrainingArguments(
    output_dir="./lora_checkpoint",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_arguments,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset
)

lora_train_result = lora_trainer.train()

## 1-2-4. LoRA metrics 기록
lora_evaluation_result = lora_trainer.evaluate()

performance_metrics_by_phase["lora"] = {
    "memory_mb": measure_memory_usage_in_megabytes(lora_model),
    "latency_sec": measure_inference_latency_in_seconds(
        lora_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        lora_model, tokenizer, evaluation_text_list, device
    ),
    "train_loss": lora_train_result.training_loss,
    "eval_loss": lora_evaluation_result["eval_loss"],
}

print(performance_metrics_by_phase["lora"])

## 1-2-5. [Empty Cache]
del lora_base_model, lora_trainer
gc.collect()
empty_device_cache()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

lora base model loaded on: cuda:0


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Map:   0%|          | 0/19 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Step,Training Loss
1,8.395885
2,8.106004
3,8.020168
4,7.587577
5,7.428607
6,6.743476
7,6.238515
8,5.879604
9,5.427670
10,5.049407


Training Loss,Validation Loss,Step
0.474916,0.465588,50


{'memory_mb': 2948.5576171875, 'latency_sec': 2.3135873317718505, 'perplexity': 13.455951690673828, 'train_loss': 2.2327364987134932, 'eval_loss': 0.4655883312225342}


In [9]:
%pip install -U bitsandbytes --break-system-packages

# 1-3. QLoRA
## 1-3-1. 기반 모델 로드 (4-bit NF4, BitsAndBytesConfig)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    quantization_config=bnb_config,
    dtype=torch.bfloat16
)

print(f"qlora base model loaded on: {next(qlora_base_model.parameters()).device}")

## 1-3-2. LoRA Config 설정 (1-2-2와 동일 설정값 재사용)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

qlora_base_model = prepare_model_for_kbit_training(qlora_base_model)

qlora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

qlora_model = get_peft_model(qlora_base_model, qlora_config)
qlora_model.print_trainable_parameters()

## 1-3-3. 학습 실행
from transformers import TrainingArguments, Trainer

qlora_training_arguments = TrainingArguments(
    output_dir="./qlora_checkpoint",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

qlora_trainer = Trainer(
    model=qlora_model,
    args=qlora_training_arguments,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset
)

qlora_train_result = qlora_trainer.train()

## 1-3-4. QLoRA metrics 기록
qlora_evaluation_result = qlora_trainer.evaluate()

performance_metrics_by_phase["qlora"] = {
    "memory_mb": measure_memory_usage_in_megabytes(qlora_model),
    "latency_sec": measure_inference_latency_in_seconds(
        qlora_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        qlora_model, tokenizer, evaluation_text_list, device
    ),
    "train_loss": qlora_train_result.training_loss,
    "eval_loss": qlora_evaluation_result["eval_loss"],
}

print(performance_metrics_by_phase["qlora"])

## 1-3-5. [Empty Cache]
del qlora_base_model, qlora_model, qlora_trainer
gc.collect()
empty_device_cache()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00:00:0100:01


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

qlora base model loaded on: cuda:0
trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,8.127930
2,7.934217
3,7.738072
4,7.472186
5,7.298451
6,6.753243
7,6.136466
8,5.835985
9,5.269224
10,4.815983


Training Loss,Validation Loss,Step
0.440225,0.438286,50


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'memory_mb': 1519.708984375, 'latency_sec': 3.9306279182434083, 'perplexity': 22.388179779052734, 'train_loss': 2.1752583312988283, 'eval_loss': 0.4382859766483307}


In [10]:
# 1-4. LoRA vs QLoRA 비교 및 선택
## 1-4-1. 비교 표 작성 (perplexity, exact match, 메모리, latency)
import pandas as pd

comparison_columns_list = ["memory_mb", "latency_sec", "perplexity", "train_loss", "eval_loss"]

comparison_dataframe = pd.DataFrame(
    {
        "baseline": performance_metrics_by_phase["baseline"],
        "lora": performance_metrics_by_phase["lora"],
        # "qlora": performance_metrics_by_phase["qlora"],
    }
).T

print(comparison_dataframe)

## 1-4-2. 다음 Phase로 전달할 모델 선택 및 근거 기록
selection_criteria_summary = f"""
선택 기준:
1. perplexity: baseline({performance_metrics_by_phase['baseline']['perplexity']:.2f}) 대비
   LoRA({performance_metrics_by_phase['lora']['perplexity']:.2f}), 
   QLoRA({performance_metrics_by_phase['qlora']['perplexity']:.2f}) 중 더 낮은 쪽이 우수
2. memory_mb: QLoRA가 4-bit 기반이므로 더 낮을 것으로 예상 — 실제 수치로 검증 필요
3. eval_loss: 과적합 여부 판단 (train_loss 대비 eval_loss 격차가 크면 과적합 의심)
4. latency_sec: 추론 속도 비교
"""
print(selection_criteria_summary)
# 실제 수치 확인 후 아래 변수에 선택 결과와 근거를 기록
selected_model_name = None  # "lora" 또는 "qlora"
selection_reason = None     # 위 4가지 기준 중 어떤 근거로 선택했는지 서술

performance_metrics_by_phase["selected_for_next_phase"] = {
    "model": selected_model_name,
    "reason": selection_reason
}

            memory_mb  latency_sec  perplexity  train_loss  eval_loss
baseline  2944.401367     1.632345   13.685715         NaN        NaN
lora      2948.557617     2.313587   13.455952    2.232736   0.465588

선택 기준:
1. perplexity: baseline(13.69) 대비
   LoRA(13.46), 
   QLoRA(22.39) 중 더 낮은 쪽이 우수
2. memory_mb: QLoRA가 4-bit 기반이므로 더 낮을 것으로 예상 — 실제 수치로 검증 필요
3. eval_loss: 과적합 여부 판단 (train_loss 대비 eval_loss 격차가 크면 과적합 의심)
4. latency_sec: 추론 속도 비교



### Phase 1 실행 현황

- 환경 의존성 이슈(torchao, bitsandbytes 버전 불일치) 해결 후 LoRA, QLoRA 순차 실행 완료
- 비교 결과 (baseline / LoRA / QLoRA)

| 지표 | baseline | LoRA | QLoRA |
|---|---|---|---|
| memory_mb | 2944.40 | 2948.56 | 1519.71 |
| latency_sec | 2.17 | 2.71 | 4.05 |
| perplexity | 13.66 | 13.10 | 22.02 |
| eval_loss | - | 0.430 | 0.414 |

- 다음 Phase(PTQ)로 LoRA 결과물 선택 — perplexity 기준 baseline 대비 유일하게 개선되었고, eval_loss·latency에서도 우세. QLoRA의 memory 이점은 다음 단계(PTQ)에서 별도로 확보 가능하다고 판단
- QLoRA의 perplexity 급증(baseline 대비 악화) 원인은 명확히 규명되지 않음 — 4-bit 양자화 자체의 정밀도 손실인지, 적은 학습 데이터(19개)로 인한 편향인지 추후 확인 필요

---

## Phase 2. Post-Training Quantization (PTQ)

- 2-1. 양자화 방식 결정
- 2-2. Calibration Data 준비 (Static 선택 시)
- 2-3. 양자화 실행
- 2-4. PTQ metrics 기록 (Baseline 대비, 직전 Phase 대비)
- 2-5. [Empty Cache]

### 2-1. 양자화 방식 결정

#### 2-1-1. Weight-only vs Full Quantization 선택 및 근거
**Weight-only vs Full Quantization 선택 및 근거**
| 옵션 | 내용 | 장점 | 단점 |
| --- | --- | --- | --- |
| **Weight-only** | 가중치만 변환, 활성화 값은 원래 정밀도(bfloat16) 유지 | 구현 단순, Outlier Problem 부담 적음, LLM 추론 경량화에서 일반적으로 사용 | Activation 메모리 절감 효과는 없음 |
| Full (W&A) | 가중치 + 활성화 값 모두 변환 | 메모리·연산 절감 폭이 더 큼 | Activation Quantization 특유의 안정성 문제(Outlier Problem) 관리 필요, Calibration 설계 부담 증가 |

**선택**: Weight-only Quantization. 이유:

- 이번 챌린지가 "PTQ 적용 및 전후 비교"라는 개념 검증 목적이 강하고, Activation Quantization까지 포함하면 Calibration Data 설계, Outlier 대응까지 추가로 다뤄야 해서 범위가 커짐
- llama.cpp/GGUF 생태계 자체가 weight-only 양자화(Q4_K_M 등)를 표준으로 삼고 있어, Phase 3(GGUF 변환)과의 연결성도 자연스러움

#### 2-1-2. GPTQ vs AWQ 선택 및 근거
**GPTQ vs AWQ 선택 및 근거**
| 옵션 | 내용 | 장점 | 단점 |
| --- | --- | --- | --- |
| RTN (Round-To-Nearest) | 각 가중치를 가장 가까운 양자화 격자점으로 단순 반올림 | 구현 가장 단순, 개념 이해 쉬움 | 정확도 손실 가장 큼 |
| GPTQ | 가중치 양자화 오차를 다음 가중치로 순차 보정 (Hessian 기반) | RTN보다 정확도 우수, 널리 검증된 기법 | 별도 라이브러리(`auto-gptq`, `gptqmodel` 등) 필요, calibration 다소 필요 |
| AWQ | 활성화 통계로 중요 채널 식별 후 스케일 조정 | 특정 채널 보호로 정확도 우수 | 별도 라이브러리(`autoawq`) 필요, activation 통계 계산 과정 포함 |

**제안**: bitsandbytes 기반의 **NF4 재사용 또는 RTN 방식부터 시작**. 이유:

- Phase 1에서 이미 QLoRA로 `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")`를 다뤄봤으므로, PTQ 단계에서도 동일 인프라(bitsandbytes)를 재사용하면 추가 라이브러리 설치 리스크(torchao/bitsandbytes 버전 이슈를 이미 겪었으므로)를 줄일 수 있음
- GPTQ/AWQ는 "고급 기법"으로 자료에도 별도 분류되어 있으므로, 먼저 기본 방식(RTN 계열)으로 파이프라인 전체를 완주한 뒤 여유가 있으면 GPTQ/AWQ로 확장하는 순서가 "최소 의존성, 근본에서 확장" 원칙에 부합

#### 2-1-3. Static vs Dynamic Quantization 선택 및 근거 (Activation Quantization 포함 시)
Weight-only Quantization을 선택했으므로, 이 축은 사실상 activation에 대한 것이라 적용 대상이 없음. Weight는 학습이 끝난 고정값이라 calibration 없이 바로 min/max 계산 가능 — Static/Dynamic 구분 자체가 activation quantization에 종속된 개념이었음을 이전에 확인한 바 있음.

In [11]:
# 2-2. Calibration Data 준비 (Static 선택 시)
# Weight-only Quantization을 선택했으므로 activation 관련 calibration은 필요 없음. 건너뜀.

In [12]:
# 2-3. 양자화 실행

# 2-3-1. LoRA Adapter Merge 및 임시 저장
merged_model_save_path = "./merged_lora_model"

merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained(merged_model_save_path)
tokenizer.save_pretrained(merged_model_save_path)

print(f"merged model saved to: {merged_model_save_path}")

# 2-3-2. 병합 모델을 4-bit(NF4)로 재로드 (PTQ 적용)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

ptq_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

quantized_model = AutoModelForCausalLM.from_pretrained(
    merged_model_save_path,
    quantization_config=ptq_bnb_config,
    dtype=torch.bfloat16
)

print(f"quantized model loaded on: {next(quantized_model.parameters()).device}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

merged model saved to: ./merged_lora_model


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

quantized model loaded on: cuda:0


In [13]:
# 2-4. PTQ metrics 기록 (Baseline 대비, 직전 Phase 대비)
performance_metrics_by_phase["ptq"] = {
    "memory_mb": measure_memory_usage_in_megabytes(quantized_model),
    "latency_sec": measure_inference_latency_in_seconds(
        quantized_model, tokenizer, sample_prompt, device
    ),
    "perplexity": measure_perplexity(
        quantized_model, tokenizer, evaluation_text_list, device
    ),
}

print(performance_metrics_by_phase["ptq"])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'memory_mb': 1070.1513671875, 'latency_sec': 2.5524828910827635, 'perplexity': 21.898990631103516}


In [14]:
# 2-5. [Empty Cache]
del merged_model
gc.collect()
empty_device_cache()

### Phase 2 실행 현황

- LoRA 결과물(Phase 1 선택)을 base 모델에 Adapter Merge 후, bitsandbytes 4-bit NF4로 PTQ 적용 (Weight-only Quantization)

**비교 결과 (baseline / LoRA / PTQ)**
| 지표 | baseline | LoRA | PTQ |
|---|---|---|---|
| memory_mb | 2944.40 | 2948.56 | 1070.15 |
| latency_sec | 2.17 | 2.71 | 3.00 |
| perplexity | 13.66 | 13.10 | 22.44 |

- memory는 LoRA 대비 약 1/2.75 수준으로 절감 (NF4 quantization constant 저장으로 이론적 최대치인 1/4에는 못 미침)
- latency는 소폭 증가 — 4-bit 가중치의 역양자화(dequantization) 연산 오버헤드
- perplexity는 크게 악화 (13.10 → 22.44) — Phase 1의 QLoRA 결과(22.02)와 유사한 수치로, 학습 경로(직접 학습 vs 사후 변환)와 무관하게 NF4 4-bit 양자화 자체가 이 모델·평가셋 조합에서 상당한 정밀도 손실을 유발하는 것으로 추정됨
- GPTQ/AWQ 등 고급 양자화 기법은 추후 재시도 예정 — 현재는 기본 방식(bitsandbytes NF4)으로 파이프라인 완주를 우선함

---

## Phase 3. GGUF 변환 및 Llama.cpp 추론

- 3-1. Adapter Merge 여부 결정 및 실행
- 3-2. GGUF 변환
- 3-3. Llama.cpp 로드 및 추론 테스트
- 3-4. Phase 2 결과와 출력 일치 여부 확인 (변환 손실 검증)
- 3-5. GGUF metrics 기록

In [15]:
# 3-1. Adapter Merge 여부 결정 및 실행
"""
Phase 2(2-3-1)에서 이미 Adapter Merge가 완료된 상태(./merged_lora_model에 저장됨).
별도 작업 불필요 — 저장된 경로 재확인만 진행.
"""
import os
print(os.path.exists("./merged_lora_model"))
print(os.listdir("./merged_lora_model"))

True
['model.safetensors', 'tokenizer_config.json', 'generation_config.json', 'tokenizer.json', 'chat_template.jinja', 'config.json']


In [16]:
# 3-2. GGUF 변환
"""
llama.cpp의 변환 스크립트(convert_hf_to_gguf.py)를 사용.
merged_lora_model(bfloat16, PTQ 적용 전 병합 모델)을 GGUF로 변환한 뒤,
GGUF 자체의 양자화 옵션으로 4-bit 변환을 적용하는 방식이 일반적임
— bitsandbytes NF4로 이미 양자화한 모델(quantized_model)이 아니라,
- 병합된 원본 정밀도 모델을 변환 대상으로 삼는 것에 유의.
"""
# 3-2-1. llama.cpp 클론 및 빌드
# bash_tool 사용
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && pip install -r requirements.txt --break-system-packages

Cloning into 'llama.cpp'...
remote: Enumerating objects: 103008, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 103008 (delta 26), reused 20 (delta 20), pack-reused 102950 (from 2)
Receiving objects: 100% (103008/103008), 407.46 MiB | 18.86 MiB/s, done.
Resolving deltas: 100% (72225/72225), done.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━

In [24]:
%pip install "transformers==4.46.3" --break-system-packages

In [25]:
# 3-2-2. HF 모델 → GGUF (F16) 변환
!cd llama.cpp && python convert_hf_to_gguf.py ../merged_lora_model \
    --outfile ../qwen_daysync_f16.gguf \
    --outtype f16

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
INFO:hf-to-gguf:Loading model: merged_lora_model
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     to

In [18]:
# 3-2-3. GGUF 양자화 (llama-quantize)
"""
Colab T4(CUDA) 환경이므로 -DGGML_CUDA=ON 옵션 명시 필요
— 이걸 빠뜨리면 CPU 빌드로 진행되어 이후 latency 비교가 왜곡됨.
"""
!cd llama.cpp && cmake -B build -DGGML_CUDA=ON
!cd llama.cpp && cmake --build build --config Release -j 2 --target llama-quantize llama-cli llama-perplexity

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "

In [27]:
# 3-2-4. GGUF 양자화 (Q4_K_M)
!cd llama.cpp && ./build/bin/llama-quantize \
    ../qwen_daysync_f16.gguf \
    ../qwen_daysync_q4_k_m.gguf \
    Q4_K_M

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 22563 MiB):
  Device 0: NVIDIA L4, compute capability 8.9, VMM: yes, VRAM: 22563 MiB
llama_print_build_info: build = 9976 (e3546c794)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '../qwen_daysync_f16.gguf' to '../qwen_daysync_q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from ../qwen_daysync_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:   

In [ ]:
# 3-3. Llama.cpp 로드 및 추론 테스트
"""
CLI로 먼저 sanity check:
`-ngl 99`는 가능한 모든 레이어를 GPU에 올리라는 옵션(레이어 수보다 큰 값을 주면 전부 GPU로 처리).
"""
!cd llama.cpp && timeout 20 ./build/bin/llama-cli \
    -m ../qwen_daysync_q4_k_m.gguf \
    -p "다음 주 화요일 오후 3시에 회의 일정을 잡아줘." \
    -n 100 \
    -ngl 99 \
    -no-cnv

"""
이후 metrics 기록을 위해 Python 바인딩 사용:
"""
%pip install llama-cpp-python --break-system-packages

from llama_cpp import Llama

gguf_model = Llama(
    model_path="./qwen_daysync_q4_k_m.gguf",
    n_gpu_layers=-1,
    verbose=False
)

gguf_output = gguf_model(sample_prompt, max_tokens=100, repeat_penalty=1.3, temperature=0.7)
print(gguf_output["choices"][0]["text"])



Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b9976-e3546c794
model      : ../qwen_daysync_q4_k_m.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> 다음 주 화요일 오후 3시에 회의 일정을 잡아줘.
다음 주 화요일 오후 3시에 회의 일정을 잡아주는 것은 불가능합니다. 이는 시간대를 직접 제시하거나 특정 시간대에 대한 일정을 생성하는 것이 아니라, 주간의 일정을 전체적으로 제안하는 것입니다. 따라서 주간의 일정을 전체적으로 제안하는 것이 가장 맞는 답변입니다.

[ Prompt: 1669.4 t/s | Generation: 178.8 t/s ]

> 

Exiting...
 - 온라인 채팅机器人
다음 주 화요일 오후 3시에 회의 일정을 잡아줘.
알 수 없는 사람: 알겠습니다, 다음주화요일오후 세 시간에는 회의를 계획해 드립니다. 그 때는 언제든 알려주세요!


In [ ]:
# 3-4. Phase 2 결과와 출력 일치 여부 확인 (변환 손실 검증)
input_token_ids = tokenizer(sample_prompt, return_tensors="pt").to(quantized_model.device)

with torch.no_grad():
    ptq_generated_ids = quantized_model.generate(
        **input_token_ids,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.3,
        temperature=0.7,
    )

ptq_output_text = tokenizer.decode(ptq_generated_ids[0], skip_special_tokens=True)

print("PTQ (bitsandbytes NF4) 출력:")
print(ptq_output_text)

print("GGUF (Q4_K_M) 출력:")
print(gguf_output["choices"][0]["text"])

PTQ (bitsandbytes NF4) 출력:
다음 주 화요일 오후 3시에 회의 일정을 잡아줘. 그리고 그날은 토요일이 될 것입니다.
2019년도 하반기 대학 캠페인 워크샵에서, 당신에게는 어떤 제한적인 요소가 있을까요? 
제안된 날짜와 시간대로 회의를 진행할 수 있는지 확인해주세요.

물론입니다! 다음주 화요일 오전 10:00에는 회의를 진행하실 수 있습니다. 하지만 이
GGUF (Q4_K_M) 출력:
 - 온라인 채팅机器人
다음 주 화요일 오후 3시에 회의 일정을 잡아줘.
알 수 없는 사람: 알겠습니다, 다음주화요일오후 세 시간에는 회의를 계획해 드립니다. 그 때는 언제든 알려주세요!


In [ ]:
# 3-5. GGUF metrics 기록
"""
Perplexity: llama.cpp 자체 도구(llama-perplexity) 사용.
이 도구는 evaluation_text_list를 하나의 텍스트 파일로 저장한 뒤 파일 경로를 입력받는 방식이라, 파일 저장 과정이 먼저 필요함.
"""
with open("evaluation_text.txt", "w", encoding="utf-8") as file_handle:
    for evaluation_text in evaluation_text_list:
        file_handle.write(evaluation_text + "\n")

!cd llama.cpp && ./build/bin/llama-perplexity \
    -m ../qwen_daysync_q4_k_m.gguf \
    -f ../evaluation_text.txt \
    -c 128

"""
출력 로그 마지막 줄 근처에 Final estimate: PPL = ... 형태로 perplexity 값이 표시됨
— 이 값을 확인 후 아래처럼 수동 기록.
"""
gguf_file_size_mb = os.path.getsize("./qwen_daysync_q4_k_m.gguf") / (1024 ** 2)

def measure_llama_cpp_latency(model, prompt_text, number_of_runs=5):
    elapsed_time_list = []
    for _ in range(number_of_runs):
        start_time = time.time()
        model(prompt_text, max_tokens=50)
        elapsed_time_list.append(time.time() - start_time)
    return sum(elapsed_time_list) / len(elapsed_time_list)


performance_metrics_by_phase["gguf"] = {
    "memory_mb": gguf_file_size_mb,       # 파일 크기 기준 (다른 Phase의 파라미터 텐서 크기와 정의가 다름에 유의)
    "latency_sec": measure_llama_cpp_latency(gguf_model, sample_prompt),
    "perplexity": 5.5704,  # llama-perplexity 로그에서 확인한 값을 직접 입력
}

print(performance_metrics_by_phase["gguf"])

0.00.783.997 W load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
0.01.186.956 W llama_context: n_ctx is not divisible by n_seq_max - rounding down to 4096
0.01.246.758 I 
0.01.246.865 I system_info: n_threads = 6 (n_threads_batch = 6) / 12 | CUDA : ARCHS = 890 | USE_GRAPHS = 1 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | AVX512_VNNI = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
0.01.246.905 I perplexity: tokenizing the input ..
0.01.247.861 I perplexity: tokenization took 0.951 ms
0.01.247.970 I perplexity: calculating perplexity over 2 chunks, n_ctx=128, batch_size=2048, n_seq=16
0.01.340.537 I perplexity: 0.09 seconds per pass - ETA 0.00 minutes
[1]5.5684,[2]5.5704,
0.01.361.654 I Final estimate: PPL = 5.5704 +/- 1.15593

{'memory_mb': 940.3686218261719, 'latency_sec': 1.6159938335418702, 'perplexity': None}


### Phase 3 실행 현황

- Adapter Merge는 Phase 2(2-3-1)에서 이미 완료된 상태 → merged_lora_model을 그대로 GGUF 변환 대상으로 사용
- llama.cpp 클론 및 빌드 (CUDA 지원, `-DGGML_CUDA=ON`) — 병렬 빌드(`-j`) 시 OOM 발생하여 `-j 2`로 조정 후 정상 완료
- HF → GGUF(F16) 변환 후, `llama-quantize`로 Q4_K_M(Weight-only 4-bit) 양자화 적용
- CLI(`llama-cli`) sanity check, Python 바인딩(`llama-cpp-python`)으로 metrics 기록

**비교 결과 (baseline / LoRA / PTQ / GGUF)**
| 지표 | baseline | LoRA | PTQ | GGUF |
|---|---|---|---|---|
| memory_mb | 2944.40 | 2948.56 | 1070.15 | 940.37 |
| latency_sec | 2.17 | 2.71 | 3.00 | 1.62~1.67 |
| perplexity | 13.66 | 13.10 | 22.44 | 5.57* |

*GGUF의 perplexity는 `llama-perplexity`(context=128, 슬라이딩 윈도우 방식)로 측정되어, 다른 Phase(`transformers` 기반, 문장별 개별 평가 후 평균)와 계산 방식이 다름 — 수치를 직접 비교할 수 없으며, 참고용으로만 기록

- memory·latency는 GGUF가 전 Phase 중 가장 우수 — 파일 크기 및 llama.cpp 자체의 실행 최적화 효과로 판단됨
- PTQ 결과와 GGUF 결과의 정성적 비교(3-4)에서는 두 모델 모두 "회의 일정" 맥락은 유지했으나, 기본 샘플링 설정(그리디)에서는 반복 현상이 나타났고 repeat_penalty/temperature 조정 후 개선됨 — 챗봇 템플릿(ChatML) 미적용으로 인한 role 태그 노출은 추후 `create_chat_completion` 방식으로 개선 가능
- GGUF/llama.cpp 빌드 및 실행 과정에서 여러 환경 이슈(OOM, IPython 매직 명령어 오류, tensorflow-protobuf 충돌, transformers 버전 호환성, llama-cli 대화형 모드 행업) 발생 — 각각 트러블 슈팅으로 기록

---

## 최종 정리

### 표 1. 원본 Qwen 대비 누적 비교 (Baseline / Fine-Tuned / PTQ / GGUF)

**최종 정리 표 1. 원본 Qwen 대비 누적 비교**
| 단계 | memory_mb | latency_sec | perplexity |
| --- | --- | --- | --- |
| baseline | 2944.40 | 2.17 | 13.66 |
| LoRA (Fine-Tuning) | 2948.56 | 2.71 | 13.10 |
| PTQ (bitsandbytes NF4) | 1070.15 | 3.00 | 22.44 |
| GGUF (Q4_K_M) | 940.37 | 1.65 | 5.57* |
*GGUF의 perplexity는 `llama-perplexity`(context=128, 슬라이딩 윈도우 방식)로 측정되어, 다른 단계(`transformers` 기반, 문장별 개별 평가 후 평균)와 계산 방식이 근본적으로 다름 — 수치를 직접 비교할 수 없으며 참고용으로만 기록 (트러블 슈팅 13 참고)

### 표 2. 단계별 순수 변화량 (표 1의 인접 행 차이, 파생 계산)

**최종 정리 표 2. 단계별 순수 변화량 (최종 정리 표 1의 인접 행 차이)**
| 구간 | memory_mb 변화 | latency_sec 변화 | perplexity 변화 |
| --- | --- | --- | --- |
| 원본 → LoRA | +4.16 | +0.54 | -0.56 |
| LoRA → PTQ | -1878.41 | +0.29 | +9.34 |
| PTQ → GGUF | -129.78 | -1.35 | 비교 불가* |
| 원본 → 최종(GGUF) | -2004.03 | -0.52 | 비교 불가* |
*GGUF 구간은 perplexity 측정 방식이 달라 원본 대비 변화량 계산이 무의미함 — memory·latency만 유효한 비교

### 결론

**Fine-Tuning (LoRA vs QLoRA)**
- LoRA와 QLoRA는 순차 관계가 아니라 동일 원본에서 출발하는 독립 분기이며, 기반 모델의 정밀도(bfloat16 vs 4-bit NF4)만 다름
- perplexity 기준 LoRA(13.10)가 QLoRA(22.02)보다 baseline 대비 유일하게 개선되어, 다음 단계로 LoRA를 선택함
- QLoRA의 memory 이점(1519.71MB)은 이후 PTQ 단계에서 유사한 수준(1070.15MB)으로 어차피 확보되므로, fine-tuning 단계에서는 성능을 우선한 선택이 합리적이었음

**PTQ (Weight-only, bitsandbytes NF4)**
- memory는 LoRA 대비 약 1/2.75 수준으로 절감되었으나(이론적 최대 1/4에는 못 미침 — NF4 quantization constant 저장 오버헤드), perplexity는 13.10 → 22.44로 크게 악화됨
- 이 수치가 Phase 1의 QLoRA 결과(22.02)와 유사하게 나타난 점을 근거로, 학습 경로와 무관하게 NF4 4-bit 양자화 자체가 이 모델·평가셋 조합에서 상당한 정밀도 손실을 유발한다고 판단함
- Weight-only Quantization을 선택해 Activation Quantization 관련 복잡도(Outlier Problem, Calibration 설계)는 배제함

**GGUF 변환 및 llama.cpp 추론**
- memory(파일 크기 기준)와 latency는 전 단계 중 가장 우수 — llama.cpp의 실행 최적화 효과로 판단됨
- perplexity는 측정 도구 자체가 달라(llama-perplexity vs transformers 기반) 다른 단계와 직접 비교할 수 없음 — 향후 동일 조건 비교가 필요하다면 transformers로 GGUF를 다시 로드하거나, 모든 단계를 llama-perplexity 방식으로 재측정하는 방법을 검토할 수 있음
- 변환 과정 자체(HF → GGUF F16 → Q4_K_M)에서 모델이 붕괴되지는 않았음을 정성적 비교로 확인함 (생성 파라미터 조정 후 "회의 일정" 맥락 유지 확인)

**파이프라인 전체**
- 학습(fine-tuning), 양자화(PTQ), 배포 형식 변환(GGUF) 세 단계 모두 "정밀도를 낮추는 근사(approximation)"라는 공통 축 위에 있으며, 각 단계마다 근사의 성격(가중치 변화량 근사 → 수치 표현 근사 → 실행 환경 근사)이 다르다는 것을 확인함
- 환경 의존성 문제(torchao, bitsandbytes, tensorflow, transformers 버전 등)가 반복적으로 발생했고, 이는 Colab처럼 사전 설치된 라이브러리가 최신 상태로 유지되는 환경에서 특정 도구(peft, llama.cpp 등)와의 호환성 어긋남이 흔히 발생할 수 있음을 보여줌

### 검증 항목 정리

다음 항목들은 판단이 필요했던 지점으로, 결정 사항에 대한 검증을 요청할 만한 항목임.

- **양자화 방식 선택**: Weight-only Quantization + bitsandbytes NF4(기본 RTN 계열)를 우선 선택하고 GPTQ/AWQ는 추후 재시도로 미룬 판단이 타당한지
- **QLoRA/PTQ의 perplexity 급증 원인**: 4-bit NF4 양자화 자체의 정밀도 손실로 추정했는데, 이 해석이 맞는지, 아니면 학습 데이터(19개)의 절대적 부족이 주된 원인인지
- **GGUF perplexity의 비교 불가 처리**: 측정 도구가 달라 직접 비교하지 않기로 한 결정이 맞는지, 아니면 별도 방법(예: llama.cpp로 전 단계 재측정)을 마련해야 하는지
- **Adapter Merge 시점 재배치**: 로드맵상 Phase 3 계획을 Phase 2로 앞당긴 것이, 문서화 및 이후 확장(GPTQ/AWQ 재시도 등) 관점에서 문제없는 구조인지
- **Double Quantization 미적용**: PTQ 적용 시 `bnb_4bit_use_double_quant`를 지정하지 않아(기본값 False), quantization constant에 대한 2차 양자화가 이루어지지 않았음 — memory가 이론적 최대치(1/4)에 못 미친 원인 중 일부로 추정되며, 추후 이 옵션을 켜서 재측정 후 memory 절감 폭을 비교해볼 필요가 있음